# Phase 5: SHAP explainability

This notebook starts fresh, so we recreate the same data, split, and
Random Forest model from Phase 4 (same random_state throughout), then
use SHAP to explain individual predictions.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, average_precision_score

df = pd.read_csv("data/raw/creditcard.csv")

X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model_rf = RandomForestClassifier(random_state=42, n_jobs=-1)
model_rf.fit(X_train, y_train)

y_scores_rf = model_rf.predict_proba(X_test)[:, 1]
y_pred_rf = (y_scores_rf >= 0.40).astype(int)  # chosen threshold from Phase 4

precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
auprc_rf = average_precision_score(y_test, y_scores_rf)

print(f"Precision: {precision_rf:.4f}  (Phase 4: 0.9333)")
print(f"Recall:    {recall_rf:.4f}  (Phase 4: 0.8571)")
print(f"AUPRC:     {auprc_rf:.4f}  (Phase 4: 0.8734)")

Precision: 0.9333  (should match Phase 4: 0.9333)
Recall:    0.8571  (should match Phase 4: 0.8571)
AUPRC:     0.8734  (should match Phase 4: 0.8734)


## Setting up SHAP

TreeExplainer is a fast, exact SHAP method built specifically for
tree-based models like Random Forest, it uses the actual tree
structure rather than approximating.

We'll start by explaining a single transaction's prediction, to see
concretely what a SHAP value represents, before scaling up.

In [2]:
import shap
import numpy as np

explainer = shap.TreeExplainer(model_rf)

# pick one transaction to explain, e.g. the first row of the test set
sample = X_test.iloc[[0]]
shap_values = explainer.shap_values(sample)

print("Shape of shap_values:", np.array(shap_values).shape)

Shape of shap_values: (1, 30, 2)
